# [Video2X](https://github.com/k4yt3x/video2x) Colab Notebook

If you encounter any issues or have any questions or suggestions, post an issue [here](https://github.com/k4yt3x/video2x/issues/new).

Licensed under the ISC License: https://www.isc.org/licenses/. \
Copyright 2022-2024 K4YT3X and contributors.

Last updated on December 15, 2024 for version [6.2.0](https://github.com/k4yt3x/video2x/releases/tag/6.2.0).

**Modified copy:** Step 1 builds Video2X from source (default: the `adeshboudhnicedigitals/video2x` fork) instead of installing the 6.2.0 release, so it includes the pipelined decode/process/encode loop and the `--queue-size` option. Step 3C compares the original serial loop with the pipelined one on a short clip.

In [ ]:
#@title ## Step 0: Runtime Selection

#@markdown **You'll need to select a GPU runtime for Video2X to work.** On the menu bar at the top, select:

#@markdown > Runtime > Change runtime type > Hardware accelerator

#@markdown Change "Hardware accelerator" to GPU and click "Save." After you're done, execute this cell to see the detailed information about the GPU allocated to you (optional).

!nvidia-smi

In [ ]:
#@title ## Step 1: Build Video2X from Source

#@markdown **This step builds Video2X from the repository and branch below instead of installing the released package.**

#@markdown - The first build takes roughly 10-20 minutes on Colab. You only need to run it once per runtime session.
#@markdown - Fork submodules (ncnn, spdlog and the model wrappers) are fetched automatically; the large Boost submodule is not needed because the system Boost is used.

repo_url = "https://github.com/adeshboudhnicedigitals/video2x.git" #@param {type:"string"}
branch = "master" #@param {type:"string"}

!add-apt-repository -y ppa:ubuntuhandbook1/ffmpeg7
!apt-get update
!DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends libvulkan1 nvidia-driver-535 ffmpeg git build-essential cmake ninja-build libavcodec-dev libavdevice-dev libavfilter-dev libavformat-dev libavutil-dev libswscale-dev libvulkan-dev glslang-tools libomp-dev libboost-program-options-dev

!rm -rf /content/video2x-src
!git clone --branch {branch} {repo_url} /content/video2x-src
!cd /content/video2x-src && git submodule update --init --recursive -- third_party/ncnn third_party/spdlog third_party/librealesrgan_ncnn_vulkan third_party/librealcugan_ncnn_vulkan third_party/librife_ncnn_vulkan

!cd /content/video2x-src && cmake -G Ninja -B build -S . -DVIDEO2X_USE_EXTERNAL_NCNN=OFF -DVIDEO2X_USE_EXTERNAL_SPDLOG=OFF -DCMAKE_CXX_COMPILER=g++ -DCMAKE_BUILD_TYPE=Release -DCMAKE_INSTALL_PREFIX=/usr
!cd /content/video2x-src && cmake --build build --config Release --target install --parallel
!ldconfig

# Confirm that the freshly built binary has the pipelining option
!video2x --help | grep -A1 "queue-size"

%env VK_ICD_FILENAMES=/usr/share/vulkan/icd.d/nvidia_icd.json

In [ ]:
#@title ## Step 2: Select/Upload Input File

#@markdown **Select or upload the file to process from Colab's local storage.**

#@markdown - ⚠️ Files in `/content` live on this runtime's local disk and are **deleted when the runtime disconnects or is reset**. Download your output before then.
#@markdown - Upload a file with the button below, or put one in `/content` yourself (for example with `!wget`) and press "Refresh".
#@markdown - To use a file at a specific path instead, fill in `custom_path`.
#@markdown - `output_dir` is where the processed video will be written.

import os
import pathlib

import ipywidgets as widgets
from google.colab import files
from IPython.display import display

custom_path = "" #@param {type:"string"}
output_dir = "/content/output" #@param {type:"string"}

media_dir = pathlib.Path("/content")
output_dir = pathlib.Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

VIDEO_SUFFIXES = {".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v", ".flv", ".wmv", ".ts"}
SKIP_DIRS = {"sample_data", "video2x-src", "pipeline_bench", "__pycache__"}

def find_videos():
    found = []
    for root, dirs, names in os.walk(media_dir):
        dirs[:] = [
            d for d in dirs
            if not d.startswith(".")
            and d not in SKIP_DIRS
            and pathlib.Path(root, d) != output_dir
        ]
        for name in names:
            if pathlib.Path(name).suffix.lower() in VIDEO_SUFFIXES:
                found.append(pathlib.Path(root, name))
    return sorted(found)

def video_options():
    return [(str(p.relative_to(media_dir)), str(p)) for p in find_videos()]

input_file = None

def select(path):
    global input_file
    input_file = pathlib.Path(path)
    print(f"Selected input file: {input_file}")

def on_dropdown_change(change):
    if change["name"] == "value" and change["new"]:
        select(change["new"])

def refresh(button=None):
    file_dropdown.options = video_options()

def upload_new_file(button):
    uploaded = files.upload()

    # Check if the user canceled the upload
    if not uploaded:
        print("Upload canceled.")
        return

    for file_name in uploaded.keys():
        path = media_dir / file_name
        refresh()
        if str(path) in [value for _, value in file_dropdown.options]:
            file_dropdown.value = str(path)
        else:
            select(path)

file_dropdown = widgets.Dropdown(
    options=video_options(),
    description="Select file:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="auto"),
)
file_dropdown.observe(on_dropdown_change)
display(file_dropdown)

refresh_button = widgets.Button(description="Refresh")
refresh_button.on_click(refresh)
upload_button = widgets.Button(description="Upload New File")
upload_button.on_click(upload_new_file)
display(widgets.HBox([refresh_button, upload_button]))

# Initial selection: a custom path wins; otherwise use what the dropdown shows
if custom_path.strip():
    custom = pathlib.Path(custom_path.strip())
    if not custom.is_file():
        raise FileNotFoundError(f"custom_path does not point to a file: {custom}")
    select(custom)
elif file_dropdown.value:
    select(file_dropdown.value)
else:
    print("No videos found in /content yet. Upload one or put one there, then press Refresh.")

In [ ]:
#@title ## Step 2B: Start Resource Monitor (optional)

#@markdown **Starts a background logger that records GPU and RAM usage while Video2X runs.** Run this cell before Step 3A/3B/3C, then run Step 4 afterwards to see the results.

#@markdown - It samples every `interval_seconds` and writes a CSV to `log_file`: GPU utilization, GPU memory, SM clock, temperature, power draw, throttle reasons, system RAM, the `video2x` process's memory and the CPU load.
#@markdown - Re-running this cell restarts the log. The logger stops when you run Step 4 with `stop_monitor` checked, or when the runtime ends.

import csv
import os
import subprocess
import threading
import time

interval_seconds = 5 #@param {type:"number"}
log_file = "/content/monitor.csv" #@param {type:"string"}

# Stop a previous monitor if this cell is re-run
if globals().get("monitor_stop") is not None:
    monitor_stop.set()
    if globals().get("monitor_thread") is not None:
        monitor_thread.join(timeout=10)

GPU_FIELDS = [
    "utilization.gpu", "utilization.memory", "memory.used", "clocks.sm",
    "clocks.max.sm", "temperature.gpu", "power.draw", "power.limit",
]

def query_gpu(fields):
    return subprocess.run(
        ["nvidia-smi", f"--query-gpu={','.join(fields)}", "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )

# The throttle-reason field was renamed in newer drivers; use whichever works
throttle_field = None
for candidate in ("clocks_throttle_reasons.active", "clocks_event_reasons.active"):
    if query_gpu(GPU_FIELDS + [candidate]).returncode == 0:
        throttle_field = candidate
        break

def num(text):
    try:
        return float(text)
    except (TypeError, ValueError):
        return None

def read_ram_mb():
    values = {}
    with open("/proc/meminfo") as f:
        for line in f:
            key, rest = line.split(":", 1)
            values[key] = int(rest.split()[0]) / 1024
    return round(values["MemTotal"] - values["MemAvailable"], 1), round(values["MemTotal"], 1)

def video2x_rss_mb():
    total = 0.0
    for pid in os.listdir("/proc"):
        if not pid.isdigit():
            continue
        try:
            with open(f"/proc/{pid}/comm") as f:
                if f.read().strip() != "video2x":
                    continue
            with open(f"/proc/{pid}/status") as f:
                for line in f:
                    if line.startswith("VmRSS:"):
                        total += int(line.split()[1]) / 1024
        except OSError:
            pass
    return round(total, 1)

HEADER = [
    "elapsed_s", "gpu_util_pct", "gpu_mem_util_pct", "gpu_mem_used_mb", "sm_clock_mhz",
    "sm_clock_max_mhz", "temp_c", "power_w", "power_limit_w", "throttle_reasons",
    "ram_used_mb", "ram_total_mb", "video2x_rss_mb", "load1",
]

def monitor_loop(stop, path, interval):
    started = time.time()
    fields = GPU_FIELDS + ([throttle_field] if throttle_field else [])
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(HEADER)
        f.flush()
        while not stop.is_set():
            gpu = [None] * 9
            result = query_gpu(fields)
            if result.returncode == 0 and result.stdout.strip():
                parts = [p.strip() for p in result.stdout.strip().splitlines()[0].split(",")]
                gpu = [num(p) for p in parts[:8]]
                gpu.append(parts[8] if len(parts) > 8 else None)
            ram_used, ram_total = read_ram_mb()
            writer.writerow(
                [round(time.time() - started, 1)] + gpu
                + [ram_used, ram_total, video2x_rss_mb(), os.getloadavg()[0]]
            )
            f.flush()
            stop.wait(interval)

monitor_stop = threading.Event()
monitor_thread = threading.Thread(
    target=monitor_loop, args=(monitor_stop, log_file, interval_seconds), daemon=True
)
monitor_thread.start()
print(f"Monitoring every {interval_seconds} s -> {log_file}")
if throttle_field is None:
    print("Note: this driver does not report throttle reasons; that column will be empty.")

In [ ]:
#@title ## Step 3A: Upscale

#@markdown **Fill in the form and run this cell to upscale a video.**

#@markdown - ⚠️ Remember free sessions can run for [at most 12 hours](https://research.google.com/colaboratory/faq.html#idle-timeouts). If you exceed this limit your progress could be lost. You can get [Colab Pro/Pro+](https://colab.research.google.com/signup/pricing) for longer runtimes.
#@markdown - It may be a good idea to upscale a small sample of your video to see how well you like the results before processing the whole video.

import pathlib
import subprocess

if 'input_file' not in globals() or input_file is None:
    raise NameError("Input file not specified. Did you forget to run the previous step?")

if 'output_dir' not in globals():
    raise NameError("Output directory not specified. Did you forget to run the previous step?")

#@markdown <br>

#@markdown ### Filter Select
#@markdown - ⚠️ `libplacebo` currently has compatibility issues with Colab's environment.
#@markdown - Both `realesrgan` and `libplacebo` can be used for anime/cartoon videos.
#@markdown - Choose `realesrgan` for real-life videos.

filter = "realesrgan" #@param ["realesrgan", "libplacebo"]

# Width and height of the output video. You can fill in one of them and the other one will be calculated automatically if it is left as `0`.
#@markdown <br>

#@markdown ### RealESRGAN Options
#@markdown - Choose `realesr-animevideov3` or `realesrgan-plus-anime` for anime/cartoon videos.
#@markdown - Choose `realesrgan-plus` for real-life videos.
#@markdown - `realesr-animevideov3` supports 2x, 3x, and 4x. Other models support only 4x.

model = "realesr-animevideov3" #@param ["realesrgan-plus-anime", "realesrgan-plus", "realesr-animevideov3"]
scale = 4 #@param {type:"slider", min:2, max:4, step:1}

#@markdown <br>

#@markdown ### libplacebo Options

#@markdown - `anime4k-v4-a`: Optimized for 1080p anime
#@markdown - `anime4k-v4-b`: Optimized for 720p anime
#@markdown - `anime4k-v4-c`: Optimized for 480p anime
#@markdown - `anime4k-v4-a+a`: Higher quality version of mode A
#@markdown - `anime4k-v4-b+b`: Higher quality version of mode B
#@markdown - `anime4k-v4-c+a`: Higher quality version of mode C
#@markdown - `anime4k-v4.1-gan`: Experimental SRGAN shaders; delivers the highest quality but is slower than standard modes

shader = "anime4k-v4-a" #@param ["anime4k-v4-a", "anime4k-v4-a+a", "anime4k-v4-b", "anime4k-v4-b+b", "anime4k-v4-c", "anime4k-v4-c+a", "anime4k-v4.1-gan"]
width = 3840 #@param {type:"number"}
height = 2160 #@param {type:"number"}

#@markdown <br>

#@markdown ### Advanced Options

#@markdown You typically don't need to touch these if you don't know what they do.

codec = "libx264" #@param {type:"string"}
preset = "slow" #@param {type:"string"}
pixfmt = "auto" #@param {type:"string"}
bitrate = 0 #@param {type:"number"}
crf = 20 #@param {type:"number"}
loglevel = "info" #@param ["trace", "debug", "info", "warn", "error", "critical", "none"]

#@markdown - `queue_size`: frames buffered between the decode, GPU and encode stages (default 4). Set it to `0` for the original, non-pipelined behavior. Each buffered 4K frame uses about 12-25 MB of RAM.
queue_size = 4 #@param {type:"integer"}

#no_copy_streams = True # @param {type:"boolean"}

output_file = output_dir / f"{input_file.stem}.processed{input_file.suffix}"
if output_file.exists():
    raise FileExistsError(f"Cannot continue: file {output_file} already exists")
print(f"Output will be written to: {output_file}")

command = [
    "video2x",

    # Common options
    "--input",
    f"\"{input_file}\"",
    "--output",
    f"\"{output_file}\"",
    "--processor",
    filter,

    # Encoder options
    "--codec",
    codec,
    "--bit-rate",
    str(bitrate),
    "--log-level",
    loglevel,
    "--queue-size",
    str(queue_size),
    "-e",
    f"preset={preset}",
    "-e",
    f"crf={crf}"
]

# RealESRGAN options
if filter == "realesrgan":
    command.extend([
        "--realesrgan-model",
        model,
        "--scaling-factor",
        str(scale)
    ])

# libplacebo options
elif filter == "libplacebo":
    command.extend([
        "--width",
        str(width),
        "--height",
        str(height),
        "--libplacebo-shader",
        f"\"{shader}\""
    ])

if pixfmt != "auto":
    command.extend([
        "--pix-fmt",
        pixfmt
    ])

!{' '.join(command)}

In [ ]:
#@title ## Step 3D: Compare Models for Picture Quality (optional)

#@markdown **Upscales a few seconds of your video with several models and settings, then shows the same frame from each side by side with the time each one took.** Pick the look you like here, then use that model in Step 3A for the whole video. Run Step 2 first.

#@markdown - The top row shows the whole frame; the bottom row zooms into the same region of each result so differences in lines, noise and blur are visible. Panels are resized to the same display size, so a result with fewer pixels will look softer, as it would on screen.
#@markdown - **Source (Lanczos)** is a plain resize of the original frame, with no AI. A model has to look better than this to be worth the time.
#@markdown - All models run at x2 here. Real-CUGAN's `conservative` keeps the original look; higher `denoise` levels remove more noise and compression artifacts but can smooth fine texture.
#@markdown - `try_720p_first` shrinks the sample to 720p before upscaling. If the real detail of your file is only 720p, it can look as good for much less GPU time.
#@markdown - Times include a few seconds of start-up per run, so they are slightly pessimistic on short samples. Real-CUGAN speeds have not been measured on this setup before; expect it to be slower than Real-ESRGAN.
#@markdown - The comparison image is also saved to `/content/output/model_comparison.png`.

import json
import pathlib
import subprocess
import time

import matplotlib.pyplot as plt
from PIL import Image

if 'input_file' not in globals() or input_file is None:
    raise NameError("Input file not specified. Did you forget to run Step 2?")

sample_start = 60 #@param {type:"number"}
sample_seconds = 2 #@param {type:"number"}
still_time = 1.0 #@param {type:"number"}
crop_center_x = 0.5 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
crop_center_y = 0.4 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
crop_width_fraction = 0.2 #@param {type:"slider", min:0.1, max:0.5, step:0.05}

try_realesrgan_animevideov3 = True #@param {type:"boolean"}
try_realcugan_conservative = True #@param {type:"boolean"}
try_realcugan_denoise1 = True #@param {type:"boolean"}
try_realcugan_denoise3 = False #@param {type:"boolean"}
try_720p_first = True #@param {type:"boolean"}

work = pathlib.Path("/content/model_comparison")
work.mkdir(exist_ok=True)
save_dir = pathlib.Path(globals().get("output_dir", "/content/output"))
save_dir.mkdir(parents=True, exist_ok=True)

def run(command, **kwargs):
    return subprocess.run(command, capture_output=True, text=True, **kwargs)

# --- Cut the sample (clamped to the input's duration) ---
duration = float(run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=nw=1:nk=1", str(input_file)],
    check=True,
).stdout.strip())
begin = sample_start
if sample_start + sample_seconds > duration:
    begin = max(0.0, duration - sample_seconds)
    print(f"The input is {duration:.1f} s long, so the sample starts at {begin:.1f} s instead of {sample_start} s.")

sample = work / "sample.mkv"
sample.unlink(missing_ok=True)
run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(begin), "-t", str(sample_seconds),
     "-i", str(input_file), "-c", "copy", "-an", str(sample)], check=True)

def probe(path):
    info = json.loads(run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0", "-count_frames",
         "-show_entries", "stream=width,height,nb_read_frames", "-of", "json", str(path)],
        check=True,
    ).stdout)["streams"][0]
    return int(info["width"]), int(info["height"]), int(info["nb_read_frames"])

src_width, src_height, src_frames = probe(sample)
if src_frames == 0:
    raise RuntimeError("The sample has no frames; check sample_start and sample_seconds")
print(f"Sample: {src_width}x{src_height}, {src_frames} frames")

still_at = min(still_time, max(0.0, sample_seconds - 0.2))

# --- Candidates: (label, input clip, extra video2x arguments) ---
sample_720 = None
if try_720p_first:
    sample_720 = work / "sample_720p.mkv"
    sample_720.unlink(missing_ok=True)
    run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(sample), "-vf", "scale=-2:720:flags=lanczos",
         "-c:v", "libx264", "-crf", "10", "-preset", "veryfast", "-an", str(sample_720)], check=True)

realesrgan = ["--processor", "realesrgan", "--realesrgan-model", "realesr-animevideov3", "--scaling-factor", "2"]
def cugan(noise):
    return ["--processor", "realcugan", "--realcugan-model", "models-se", "--scaling-factor", "2",
            f"--noise-level={noise}"]

candidates = []
if try_realesrgan_animevideov3:
    candidates.append(("Real-ESRGAN animevideov3", sample, realesrgan))
if try_realcugan_conservative:
    candidates.append(("Real-CUGAN conservative", sample, cugan(-1)))
if try_realcugan_denoise1:
    candidates.append(("Real-CUGAN denoise 1", sample, cugan(1)))
if try_realcugan_denoise3:
    candidates.append(("Real-CUGAN denoise 3", sample, cugan(3)))
if try_720p_first:
    candidates.append(("animevideov3 from 720p", sample_720, realesrgan))
if not candidates:
    raise ValueError("Select at least one model to compare.")

# --- Run each candidate and grab the same frame ---
results = []
for index, (label, clip, arguments) in enumerate(candidates):
    out = work / f"result_{index}.mkv"
    out.unlink(missing_ok=True)
    print(f"Running {label}...")
    started = time.time()
    result = run(
        ["video2x", "--input", str(clip), "--output", str(out), "--codec", "libx264",
         "-e", "preset=veryfast", "-e", "crf=14", "--log-level", "warn", "--no-progress",
         "--queue-size", "4"] + arguments
    )
    seconds = time.time() - started
    if result.returncode != 0 or not out.exists():
        print(f"  failed, skipping. Output tail:\n{(result.stdout + result.stderr)[-800:]}")
        continue
    still = work / f"still_{index}.png"
    still.unlink(missing_ok=True)
    run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(still_at), "-i", str(out),
         "-frames:v", "1", str(still)], check=True)
    results.append((f"{label}\n{seconds:.0f} s ({src_frames / seconds:.2f} fps)", Image.open(still).convert("RGB")))
    print(f"  done in {seconds:.1f} s")

if not results:
    raise RuntimeError("No model produced a result.")

# --- Plain Lanczos resize of the source frame as the no-AI reference ---
ref_width, ref_height = results[0][1].size
reference_path = work / "still_reference.png"
reference_path.unlink(missing_ok=True)
run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(still_at), "-i", str(sample), "-frames:v", "1",
     "-vf", f"scale={ref_width}:{ref_height}:flags=lanczos", str(reference_path)], check=True)
panels = [("Source (Lanczos)\nno AI", Image.open(reference_path).convert("RGB"))] + results

# --- Show whole frames (top) and zoomed crops (bottom) ---
display_width = 640
def crop_region(image):
    width, height = image.size
    crop_w = int(width * crop_width_fraction)
    crop_h = int(crop_w * height / width)
    left = min(max(int(width * crop_center_x - crop_w / 2), 0), width - crop_w)
    top = min(max(int(height * crop_center_y - crop_h / 2), 0), height - crop_h)
    return image.crop((left, top, left + crop_w, top + crop_h))

figure, axes = plt.subplots(2, len(panels), figsize=(4.2 * len(panels), 5.6), squeeze=False)
for column, (title, image) in enumerate(panels):
    full = image.resize((display_width, int(display_width * image.height / image.width)), Image.LANCZOS)
    zoom = crop_region(image)
    zoom = zoom.resize((display_width, int(display_width * zoom.height / zoom.width)), Image.BICUBIC)
    axes[0][column].imshow(full)
    axes[0][column].set_title(title, fontsize=9)
    axes[1][column].imshow(zoom)
    for row in (0, 1):
        axes[row][column].axis("off")
figure.tight_layout()
comparison_path = save_dir / "model_comparison.png"
figure.savefig(comparison_path, dpi=150)
plt.show()
print(f"Saved {comparison_path}")

In [ ]:
#@title ## Step 3C: Compare Pipelining Speed (optional)

#@markdown **Cuts a short clip from the selected input and upscales it twice: once with `--queue-size 0` (original serial loop) and once with the queue size below. Then compares the run times and checks that both outputs contain identical frames.**

#@markdown - Run Step 2 first so an input file is selected.
#@markdown - The clip is cut without re-encoding, so it starts at the nearest keyframe before `clip_start`.
#@markdown - If the frame hashes differ, that may come from the encoder rather than the pipeline; compare the run times either way.

import pathlib
import subprocess
import time

if 'input_file' not in globals() or input_file is None:
    raise NameError("Input file not specified. Did you forget to run Step 2?")

clip_start = 60 #@param {type:"number"}
clip_seconds = 30 #@param {type:"number"}
pipelined_queue_size = 4 #@param {type:"integer"}
model = "realesr-animevideov3" #@param ["realesrgan-plus-anime", "realesrgan-plus", "realesr-animevideov3"]
scale = 2 #@param {type:"slider", min:2, max:4, step:1}
preset = "slow" #@param {type:"string"}
crf = 20 #@param {type:"number"}

work = pathlib.Path("/content/pipeline_bench")
work.mkdir(exist_ok=True)
clip = work / f"clip{input_file.suffix}"
clip.unlink(missing_ok=True)

duration = float(subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=nw=1:nk=1", str(input_file)],
    capture_output=True, text=True, check=True,
).stdout.strip())
clip_begin = clip_start
if clip_start + clip_seconds > duration:
    clip_begin = max(0.0, duration - clip_seconds)
    print(f"The input is {duration:.1f} s long, so the clip starts at {clip_begin:.1f} s instead of {clip_start} s.")

subprocess.run(
    ["ffmpeg", "-y", "-loglevel", "error", "-ss", str(clip_begin), "-t", str(clip_seconds),
     "-i", str(input_file), "-c", "copy", str(clip)],
    check=True,
)

def run(queue_size):
    out = work / f"out_q{queue_size}.mkv"
    out.unlink(missing_ok=True)
    command = [
        "video2x", "--input", str(clip), "--output", str(out),
        "--processor", "realesrgan", "--realesrgan-model", model,
        "--scaling-factor", str(scale),
        "--codec", "libx264", "-e", f"preset={preset}", "-e", f"crf={crf}",
        "--log-level", "warn", "--no-progress",
        "--queue-size", str(queue_size),
    ]
    start = time.time()
    subprocess.run(command, check=True)
    return out, time.time() - start

def frame_lines(path):
    result = subprocess.run(
        ["ffmpeg", "-loglevel", "error", "-i", str(path), "-map", "0:v", "-f", "framemd5", "-"],
        capture_output=True, text=True, check=True,
    )
    return [line.strip() for line in result.stdout.splitlines() if line and not line.startswith("#")]

print("Running serial (--queue-size 0)...")
serial_out, serial_time = run(0)
print(f"Running pipelined (--queue-size {pipelined_queue_size})...")
pipelined_out, pipelined_time = run(pipelined_queue_size)

serial_frames = frame_lines(serial_out)
pipelined_frames = frame_lines(pipelined_out)

print()
print(f"Serial:    {serial_time:7.1f} s  ({len(serial_frames)} frames)")
print(f"Pipelined: {pipelined_time:7.1f} s  ({len(pipelined_frames)} frames)")
print(f"Speedup:   {serial_time / pipelined_time:.2f}x")
print("Frame count, timestamps and pixel hashes:",
      "IDENTICAL" if serial_frames == pipelined_frames else "DIFFERENT")

In [ ]:
#@title ## Step 3B: Frame Interpolation 🚧 BROKEN 🚧

#@markdown **Fill in the form and run this cell to upscale a video.**

#@markdown - ⚠️ RIFE is currently broken on Colab for unknown reasons. SPIRV will fail to compile.
#@markdown - ⚠️ Remember free sessions can run for [at most 12 hours](https://research.google.com/colaboratory/faq.html#idle-timeouts). If you exceed this limit your progress could be lost. You can get [Colab Pro/Pro+](https://colab.research.google.com/signup/pricing) for longer runtimes.
#@markdown - It may be a good idea to upscale a small sample of your video to see how well you like the results before processing the whole video.

import pathlib
import subprocess

if 'input_file' not in globals() or input_file is None:
    raise NameError("Input file not specified. Did you forget to run the previous step?")

if 'output_dir' not in globals():
    raise NameError("Output directory not specified. Did you forget to run the previous step?")

#@markdown <br>

#@markdown ### Interpolator Select
#@markdown - Choose `rife` for real-life videos.

interpolator = "rife" #@param ["rife"]

# Width and height of the output video. You can fill in one of them and the other one will be calculated automatically if it is left as `0`.
#@markdown <br>

#@markdown ### Rife Options

model = "rife-v4.6" #@param ["rife", "rife-HD", "rife-UHD", "rife-anime", "rife-v2", "rife-v2.3", "rife-v2.4", "rife-v3.0", "rife-v3.1", "rife-v4", "rife-v4.6"]
frm_rate_mul = 2 #@param {type:"slider", min:2, max:10, step:1}

#@markdown <br>

#@markdown ### Advanced Options

#@markdown You typically don't need to touch these if you don't know what they do.

codec = "libx264" #@param {type:"string"}
preset = "slow" #@param {type:"string"}
pixfmt = "auto" #@param {type:"string"}
bitrate = 0 #@param {type:"number"}
crf = 20 #@param {type:"number"}
loglevel = "info" #@param ["trace", "debug", "info", "warn", "error", "critical", "none"]

#@markdown - `queue_size`: frames buffered between the decode, GPU and encode stages (default 4). Set it to `0` for the original, non-pipelined behavior. Each buffered 4K frame uses about 12-25 MB of RAM.
queue_size = 4 #@param {type:"integer"}

#no_copy_streams = True # @param {type:"boolean"}

output_file = output_dir / f"{input_file.stem}.processed{input_file.suffix}"
if output_file.exists():
    raise FileExistsError(f"Cannot continue: file {output_file} already exists")
print(f"Output will be written to: {output_file}")

command = [
    "video2x",

    # Common options
    "--input",
    f"\"{input_file}\"",
    "--output",
    f"\"{output_file}\"",
    "--processor",
    interpolator,

    # Encoder options
    "--codec",
    codec,
    "--bit-rate",
    str(bitrate),
    "--log-level",
    loglevel,
    "--queue-size",
    str(queue_size),
    "-e",
    f"preset={preset}",
    "-e",
    f"crf={crf}"
]

# RIFE options
if interpolator == "rife":
    command.extend([
        "--rife-model",
        model,
        "--frame-rate-mul",
        str(frm_rate_mul)
    ])

if pixfmt != "auto":
    command.extend([
        "--pix-fmt",
        pixfmt
    ])

!{' '.join(command)}

In [ ]:
#@title ## Step 4: Resource Monitor Results (optional)

#@markdown **Summarizes and plots what the Step 2B monitor recorded.** Run it after the job finishes (Colab runs one cell at a time, so it cannot run while Step 3 is still going). The log keeps being written while the job runs, and is also readable with `!tail /content/monitor.csv` from a Colab terminal if you have one.

#@markdown - The table compares the first, middle and last third of the run, so a steady decline shows up as a trend.
#@markdown - Check `stop_monitor` to stop logging after showing the results.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

stop_monitor = False #@param {type:"boolean"}

if "log_file" not in globals():
    raise NameError("No monitor log found. Did you run Step 2B?")

df = pd.read_csv(log_file)
for column in df.columns:
    if column != "throttle_reasons":
        df[column] = pd.to_numeric(df[column], errors="coerce")

if len(df) < 3:
    raise RuntimeError(f"Only {len(df)} samples so far; let the job run a little longer.")

df["minutes"] = df["elapsed_s"] / 60
print(f"{len(df)} samples over {df['minutes'].iloc[-1]:.1f} minutes")

# Keep only the samples taken while video2x was running, so idle time before or
# after the job does not distort the comparison
active = df[df["video2x_rss_mb"] > 0]
if len(active) >= 3:
    print(f"{len(active)} of {len(df)} samples were taken while video2x was running; using those.")
    df = active.reset_index(drop=True)
else:
    print("video2x was not seen running; using all samples.")

# Compare thirds of the run
columns = ["gpu_util_pct", "sm_clock_mhz", "temp_c", "power_w",
           "ram_used_mb", "video2x_rss_mb", "load1"]
edges = np.linspace(0, len(df), 4, dtype=int)
thirds = [df.iloc[edges[i]:edges[i + 1]] for i in range(3)]
summary = pd.DataFrame(
    {name: part[columns].mean().round(1) for name, part in
     zip(["first third", "middle third", "last third"], thirds)}
).T
display(summary)

for column, label in (("sm_clock_mhz", "SM clock"), ("ram_used_mb", "System RAM used"),
                      ("video2x_rss_mb", "video2x memory"), ("gpu_util_pct", "GPU utilization")):
    first, last = summary.loc["first third", column], summary.loc["last third", column]
    if pd.notna(first) and pd.notna(last) and first:
        print(f"{label}: {first:g} -> {last:g} ({(last - first) / first * 100:+.0f}%)")

# Decode GPU throttle reasons
THROTTLE_BITS = {
    0x1: "GPU idle", 0x2: "application clocks setting", 0x4: "software power cap",
    0x8: "hardware slowdown", 0x10: "sync boost", 0x20: "software thermal slowdown",
    0x40: "hardware thermal slowdown", 0x80: "hardware power brake",
    0x100: "display clock setting",
}
counts = {}
for value in df["throttle_reasons"].dropna():
    try:
        mask = int(str(value), 16)
    except ValueError:
        continue
    for bit, name in THROTTLE_BITS.items():
        if mask & bit:
            counts[name] = counts.get(name, 0) + 1
if counts:
    print("Throttle reasons (samples):", ", ".join(f"{k}: {v}" for k, v in sorted(counts.items())))
else:
    print("No throttle reasons reported.")

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
axes[0].plot(df["minutes"], df["gpu_util_pct"], label="GPU utilization (%)")
axes[0].plot(df["minutes"], df["sm_clock_mhz"] / df["sm_clock_max_mhz"].max() * 100,
             label="SM clock (% of max)")
axes[0].set_ylabel("%")
axes[0].legend()
axes[1].plot(df["minutes"], df["temp_c"], label="Temperature (C)")
axes[1].plot(df["minutes"], df["power_w"], label="Power (W)")
axes[1].legend()
axes[2].plot(df["minutes"], df["ram_used_mb"], label="System RAM used (MB)")
axes[2].plot(df["minutes"], df["video2x_rss_mb"], label="video2x memory (MB)")
axes[2].set_xlabel("Minutes")
axes[2].legend()
plt.tight_layout()
plt.show()

if stop_monitor and globals().get("monitor_stop") is not None:
    monitor_stop.set()
    print("Monitor stopped.")

In [ ]:
#@title ## Step 5A: Prepare Backend Feasibility Test (experimental)

#@markdown **Experiment, separate from Video2X's own backends.** Steps 5A and 5B measure how fast the same Real-ESRGAN `realesr-animevideov3` network runs through PyTorch/CUDA (and optionally ONNX Runtime) compared with Video2X's ncnn-Vulkan path.

#@markdown - 5A downloads the official weights, defines the network and loads it. Run it once.
#@markdown - The official weights are the **x4** model. Most of the network's work happens at the input resolution, so its speed is comparable to x2, but not identical.
#@markdown - `install_onnxruntime` also installs `onnx` and `onnxruntime-gpu` (used by the optional ONNX Runtime test in 5B).

import pathlib
import subprocess
import sys
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F

install_onnxruntime = True #@param {type:"boolean"}

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU available. Select a GPU runtime in Step 0.")
print(f"GPU: {torch.cuda.get_device_name(0)}   torch {torch.__version__}   CUDA {torch.version.cuda}")

if install_onnxruntime:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime-gpu"], check=False
    )

weights_url = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth"
weights_path = pathlib.Path("/content/realesr-animevideov3.pth")
if not weights_path.exists():
    urllib.request.urlretrieve(weights_url, weights_path)

class SRVGGNetCompact(nn.Module):
    def __init__(self, num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=16, upscale=4):
        super().__init__()
        self.upscale = upscale
        self.body = nn.ModuleList()
        self.body.append(nn.Conv2d(num_in_ch, num_feat, 3, 1, 1))
        self.body.append(nn.PReLU(num_parameters=num_feat))
        for _ in range(num_conv):
            self.body.append(nn.Conv2d(num_feat, num_feat, 3, 1, 1))
            self.body.append(nn.PReLU(num_parameters=num_feat))
        self.body.append(nn.Conv2d(num_feat, num_out_ch * upscale * upscale, 3, 1, 1))
        self.upsampler = nn.PixelShuffle(upscale)

    def forward(self, x):
        out = x
        for layer in self.body:
            out = layer(out)
        out = self.upsampler(out)
        return out + F.interpolate(x, scale_factor=self.upscale, mode="nearest")

checkpoint = torch.load(weights_path, map_location="cpu")
state_dict = checkpoint["params"] if "params" in checkpoint else checkpoint
feasibility_model = SRVGGNetCompact()
feasibility_model.load_state_dict(state_dict, strict=True)
feasibility_model.eval()

parameter_count = sum(p.numel() for p in feasibility_model.parameters())
print(f"Loaded realesr-animevideov3 (x4): {parameter_count / 1e6:.2f} M parameters")

In [ ]:
#@title ## Step 5B: Run Backend Comparison (experimental)

#@markdown **Compares Video2X's ncnn-Vulkan path with PyTorch/CUDA (and optionally ONNX Runtime) running the same network on your GPU.** Run Step 2 and Step 5A first.

#@markdown - **Baseline:** `video2x --benchmark` on a short clip cut from your input (scale 4, matching the x4 weights). It skips encoding, but still includes decoding, colour conversion and tiling. The speed is read from the summary that `video2x` prints at the `info` log level.
#@markdown - **Variants:** raw network time on one 1080p-sized frame, with no video decode, conversion or encode. This is a best case, so treat the speedup as an upper bound.
#@markdown - **Decision rule:** about 2x or more over the baseline is worth considering for integration; about 1.3x or less is not.
#@markdown - GPU clocks fall as the T4 heats up, so the SM clock and temperature are printed after each measurement.

import copy
import pathlib
import re
import subprocess
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display

if 'input_file' not in globals() or input_file is None:
    raise NameError("Input file not specified. Did you forget to run Step 2?")
if 'feasibility_model' not in globals():
    raise NameError("Network not loaded. Did you forget to run Step 5A?")

clip_start = 60 #@param {type:"number"}
clip_seconds = 5 #@param {type:"number"}
frames_to_time = 60 #@param {type:"integer"}
frame_height = 1080 #@param {type:"integer"}
frame_width = 1920 #@param {type:"integer"}
run_onnxruntime = True #@param {type:"boolean"}

work = pathlib.Path("/content/pipeline_bench")
work.mkdir(exist_ok=True)

def gpu_state():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=clocks.sm,temperature.gpu,power.draw", "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    return result.stdout.strip().replace("\n", " ")

results = []

# --- Baseline: Video2X ncnn-Vulkan, benchmark mode (no encoding) ---
clip = work / f"feasibility_clip{input_file.suffix}"
clip.unlink(missing_ok=True)
duration = float(subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=nw=1:nk=1", str(input_file)],
    capture_output=True, text=True, check=True,
).stdout.strip())
clip_begin = clip_start
if clip_start + clip_seconds > duration:
    clip_begin = max(0.0, duration - clip_seconds)
    print(f"The input is {duration:.1f} s long, so the clip starts at {clip_begin:.1f} s instead of {clip_start} s.")

subprocess.run(
    ["ffmpeg", "-y", "-loglevel", "error", "-ss", str(clip_begin), "-t", str(clip_seconds),
     "-i", str(input_file), "-c", "copy", str(clip)],
    check=True,
)
bench_out = work / "feasibility_bench_out.mkv"
bench = subprocess.run(
    ["video2x", "--input", str(clip), "--output", str(bench_out),
     "--processor", "realesrgan", "--realesrgan-model", "realesr-animevideov3",
     "--scaling-factor", "4", "--benchmark", "--log-level", "info", "--no-progress"],
    capture_output=True, text=True,
)
bench_out.unlink(missing_ok=True)
match = re.search(r"Average processing speed:\s*([0-9.]+)\s*FPS", bench.stdout + bench.stderr)
if not match:
    print("Could not read the baseline speed from video2x. Its output was:")
    print((bench.stdout + bench.stderr)[-1500:])
    raise RuntimeError("Baseline run failed")
baseline_fps = float(match.group(1))
if baseline_fps <= 0:
    print("video2x reported 0 FPS, so it processed no frames. Its output was:")
    print((bench.stdout + bench.stderr)[-1500:])
    raise RuntimeError("Baseline processed no frames; check the clip (is the input long enough?)")
results.append(("ncnn Vulkan (video2x --benchmark, x4)", 1000 / baseline_fps, baseline_fps, None, gpu_state()))
print(f"Baseline: {baseline_fps:.2f} fps   GPU: {gpu_state()}")

# --- PyTorch variants ---
def time_torch(dtype, channels_last, frames):
    model = copy.deepcopy(feasibility_model).to("cuda").to(dtype).eval()
    x = torch.rand(1, 3, frame_height, frame_width, device="cuda", dtype=dtype)
    if channels_last:
        model = model.to(memory_format=torch.channels_last)
        x = x.contiguous(memory_format=torch.channels_last)
    torch.backends.cudnn.benchmark = True
    torch.cuda.reset_peak_memory_stats()
    with torch.inference_mode():
        for _ in range(5):
            model(x)
        torch.cuda.synchronize()
        started = time.time()
        for _ in range(frames):
            model(x)
        torch.cuda.synchronize()
        seconds = time.time() - started
    peak_mb = torch.cuda.max_memory_allocated() / 1024 / 1024
    del model, x
    torch.cuda.empty_cache()
    return seconds / frames, peak_mb

variants = [
    ("PyTorch fp32", torch.float32, False, max(10, frames_to_time // 4)),
    ("PyTorch fp16", torch.float16, False, frames_to_time),
    ("PyTorch fp16 + channels_last", torch.float16, True, frames_to_time),
]
for name, dtype, channels_last, frames in variants:
    seconds_per_frame, peak_mb = time_torch(dtype, channels_last, frames)
    results.append((name, seconds_per_frame * 1000, 1 / seconds_per_frame, peak_mb, gpu_state()))
    print(f"{name}: {1 / seconds_per_frame:.2f} fps   GPU: {gpu_state()}")

# --- ONNX Runtime (optional) ---
if run_onnxruntime:
    try:
        import onnxruntime as ort
        if hasattr(ort, "preload_dlls"):
            ort.preload_dlls()
        model = copy.deepcopy(feasibility_model).to("cuda").half().eval()
        x = torch.rand(1, 3, frame_height, frame_width, device="cuda", dtype=torch.float16)
        onnx_path = str(work / "animevideov3_fp16.onnx")
        export_args = dict(input_names=["input"], output_names=["output"], opset_version=17)
        try:
            torch.onnx.export(model, x, onnx_path, dynamo=False, **export_args)
        except TypeError:
            torch.onnx.export(model, x, onnx_path, **export_args)
        session = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider"])
        if "CUDAExecutionProvider" not in session.get_providers():
            raise RuntimeError("ONNX Runtime fell back to CPU (CUDA provider not available)")
        out = torch.empty(1, 3, frame_height * 4, frame_width * 4, device="cuda", dtype=torch.float16)
        binding = session.io_binding()
        binding.bind_input("input", "cuda", 0, np.float16, tuple(x.shape), x.data_ptr())
        binding.bind_output("output", "cuda", 0, np.float16, tuple(out.shape), out.data_ptr())
        for _ in range(5):
            session.run_with_iobinding(binding)
        binding.synchronize_outputs()
        started = time.time()
        for _ in range(frames_to_time):
            session.run_with_iobinding(binding)
        binding.synchronize_outputs()
        seconds_per_frame = (time.time() - started) / frames_to_time
        results.append(("ONNX Runtime CUDA fp16", seconds_per_frame * 1000, 1 / seconds_per_frame, None, gpu_state()))
        print(f"ONNX Runtime CUDA fp16: {1 / seconds_per_frame:.2f} fps   GPU: {gpu_state()}")
        del model, x, out, session, binding
        torch.cuda.empty_cache()
    except Exception as error:
        print(f"ONNX Runtime test skipped: {type(error).__name__}: {str(error)[:300]}")

table = pd.DataFrame(results, columns=["backend", "ms_per_frame", "fps", "peak_vram_mb", "gpu_clock_temp_power"])
table["speedup_vs_baseline"] = (table["fps"] / baseline_fps).round(2)
table["ms_per_frame"] = table["ms_per_frame"].round(1)
table["fps"] = table["fps"].round(2)
table["peak_vram_mb"] = table["peak_vram_mb"].round(0)
display(table)

best = table.iloc[1:]["speedup_vs_baseline"].max()
print()
if best >= 2:
    print(f"Best raw-network speedup: {best:.1f}x. Worth considering integration (still an upper bound: it excludes video I/O).")
elif best <= 1.3:
    print(f"Best raw-network speedup: {best:.1f}x. Not worth a new backend.")
else:
    print(f"Best raw-network speedup: {best:.1f}x. Marginal; an end-to-end test would be needed to decide.")

In [ ]:
#@title ## Step 5C: End-to-End CUDA Pipeline Test (experimental)

#@markdown **Runs the same short clip through (1) Video2X's normal ncnn-Vulkan path and (2) a PyTorch/CUDA pipeline, both with real encoding, and compares wall-clock time and output quality.** Run Steps 2, 5A and 5B first.

#@markdown - Both use scale 4 (the official weights are x4) and the same `codec`, `preset` and `crf`. Audio is not copied in this test, since it does not affect speed.
#@markdown - The CUDA pipeline decodes with ffmpeg, runs the network on the GPU in fp16 with `channels_last`, and pipes raw RGB frames back to an ffmpeg encoder, using three threads (decode, GPU, encode).
#@markdown - The CPU does the RGB to YUV conversion inside ffmpeg here, as Video2X's own path does with swscale, so that cost is in both runs.
#@markdown - The Video2X run includes about a few seconds of start-up and model loading; the CUDA run's model warm-up is excluded. Keep that in mind on short clips.
#@markdown - `mode`: **with encoding** measures the real end-to-end job. **no encoding** skips encoding on both sides (Video2X runs with `--benchmark`, the CUDA pipeline writes to a null muxer) to compare the GPU paths themselves. At 8K output the CPU encoder can be the shared bottleneck, which hides any difference, so run both.
#@markdown - `compare_quality` computes PSNR between the two outputs as a sanity check. Different weights, tiling or colour handling would show up as a low value.

import copy
import json
import pathlib
import queue
import re
import subprocess
import threading
import time

import pandas as pd
import torch
from IPython.display import display

if 'input_file' not in globals() or input_file is None:
    raise NameError("Input file not specified. Did you forget to run Step 2?")
if 'feasibility_model' not in globals():
    raise NameError("Network not loaded. Did you forget to run Step 5A?")

clip_start = 0 #@param {type:"number"}
clip_seconds = 5 #@param {type:"number"}
codec = "libx264" #@param {type:"string"}
preset = "veryfast" #@param {type:"string"}
crf = 20 #@param {type:"number"}
queue_size = 4 #@param {type:"integer"}
mode = "with encoding" #@param ["with encoding", "no encoding"]
compare_quality = True #@param {type:"boolean"}

work = pathlib.Path("/content/pipeline_bench")
work.mkdir(exist_ok=True)

# --- Cut the clip (clamped to the input's duration) ---
duration = float(subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=nw=1:nk=1", str(input_file)],
    capture_output=True, text=True, check=True,
).stdout.strip())
clip_begin = clip_start
if clip_start + clip_seconds > duration:
    clip_begin = max(0.0, duration - clip_seconds)
    print(f"The input is {duration:.1f} s long, so the clip starts at {clip_begin:.1f} s instead of {clip_start} s.")

clip = work / f"e2e_clip{input_file.suffix}"
clip.unlink(missing_ok=True)
subprocess.run(
    ["ffmpeg", "-y", "-loglevel", "error", "-ss", str(clip_begin), "-t", str(clip_seconds),
     "-i", str(input_file), "-c", "copy", "-an", str(clip)],
    check=True,
)

probe = json.loads(subprocess.run(
    ["ffprobe", "-v", "error", "-select_streams", "v:0", "-count_frames",
     "-show_entries", "stream=width,height,r_frame_rate,nb_read_frames", "-of", "json", str(clip)],
    capture_output=True, text=True, check=True,
).stdout)["streams"][0]
width, height = int(probe["width"]), int(probe["height"])
frame_rate = probe["r_frame_rate"]
expected_frames = int(probe["nb_read_frames"])
if expected_frames == 0:
    raise RuntimeError("The clip has no frames; check clip_start and clip_seconds")
print(f"Clip: {width}x{height}, {frame_rate} fps, {expected_frames} frames -> {width * 4}x{height * 4} output")

with_encoding = mode == "with encoding"
encoder_args = ["-c:v", codec, "-preset", preset, "-crf", str(crf), "-pix_fmt", "yuv420p"]

# --- 1. Video2X ncnn-Vulkan path (real run, with encoding) ---
ncnn_out = work / "e2e_ncnn.mp4"
ncnn_out.unlink(missing_ok=True)
print("Running Video2X (ncnn Vulkan)...")
started = time.time()
ncnn_command = [
    "video2x", "--input", str(clip), "--output", str(ncnn_out),
    "--processor", "realesrgan", "--realesrgan-model", "realesr-animevideov3",
    "--scaling-factor", "4", "--codec", codec,
    "-e", f"preset={preset}", "-e", f"crf={crf}",
    "--log-level", "warn", "--no-progress", "--queue-size", str(queue_size),
]
if not with_encoding:
    ncnn_command.append("--benchmark")
ncnn = subprocess.run(ncnn_command, capture_output=True, text=True)
ncnn_seconds = time.time() - started
if not with_encoding:
    ncnn_out.unlink(missing_ok=True)
if ncnn.returncode != 0:
    print((ncnn.stdout + ncnn.stderr)[-1500:])
    raise RuntimeError("The Video2X run failed")
print(f"  done in {ncnn_seconds:.1f} s")

# --- 2. PyTorch/CUDA pipeline ---
cuda_out = work / "e2e_cuda.mp4"
cuda_out.unlink(missing_ok=True)
out_width, out_height = width * 4, height * 4
frame_bytes = width * height * 3

model = copy.deepcopy(feasibility_model).to("cuda").half().to(memory_format=torch.channels_last).eval()
torch.backends.cudnn.benchmark = True
with torch.inference_mode():
    warmup = torch.zeros(1, 3, height, width, device="cuda", dtype=torch.float16)
    warmup = warmup.contiguous(memory_format=torch.channels_last)
    model(warmup)
    model(warmup)
    torch.cuda.synchronize()
del warmup

decoder = subprocess.Popen(
    ["ffmpeg", "-loglevel", "error", "-i", str(clip), "-f", "rawvideo", "-pix_fmt", "rgb24", "-"],
    stdout=subprocess.PIPE,
)
encoder_command = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24",
                   "-s", f"{out_width}x{out_height}", "-r", frame_rate, "-i", "-"]
if with_encoding:
    encoder_command += encoder_args + [str(cuda_out)]
else:
    encoder_command += ["-f", "null", "-"]
encoder = subprocess.Popen(encoder_command, stdin=subprocess.PIPE)

read_q = queue.Queue(maxsize=queue_size)
write_q = queue.Queue(maxsize=queue_size)
free_q = queue.Queue()
for _ in range(queue_size + 3):
    free_q.put(torch.empty((out_height, out_width, 3), dtype=torch.uint8).pin_memory())
errors = []

def reader():
    try:
        while True:
            data = decoder.stdout.read(frame_bytes)
            if len(data) < frame_bytes:
                break
            read_q.put(bytearray(data))
    except Exception as error:
        errors.append(error)
    finally:
        read_q.put(None)

def writer():
    broken = False
    while True:
        item = write_q.get()
        if item is None:
            break
        if not broken:
            try:
                encoder.stdin.write(memoryview(item.numpy()))
            except Exception as error:
                errors.append(error)
                broken = True
        free_q.put(item)

reader_thread = threading.Thread(target=reader, daemon=True)
writer_thread = threading.Thread(target=writer, daemon=True)

print("Running PyTorch/CUDA pipeline...")
frames = 0
wait_input = 0.0
wait_output = 0.0
wait_encoder = 0.0
started = time.time()
reader_thread.start()
writer_thread.start()
try:
    with torch.inference_mode():
        while True:
            mark = time.time()
            item = read_q.get()
            wait_input += time.time() - mark
            if item is None or errors:
                break
            x = torch.frombuffer(item, dtype=torch.uint8).view(height, width, 3).to("cuda")
            x = x.permute(2, 0, 1).unsqueeze(0).half().div_(255)
            x = x.contiguous(memory_format=torch.channels_last)
            y = model(x)
            y = y.mul_(255).round_().clamp_(0, 255).to(torch.uint8)
            y = y.permute(0, 2, 3, 1).contiguous()
            mark = time.time()
            buffer = free_q.get()
            wait_output += time.time() - mark
            buffer.copy_(y[0])
            mark = time.time()
            write_q.put(buffer)
            wait_encoder += time.time() - mark
            frames += 1
finally:
    write_q.put(None)
    writer_thread.join()
    try:
        encoder.stdin.close()
    except Exception:
        pass
    encoder.wait()
    decoder.stdout.close()
    decoder.wait()
cuda_seconds = time.time() - started

if errors or encoder.returncode != 0:
    raise RuntimeError(f"The CUDA pipeline failed: {errors or 'ffmpeg encoder exit code ' + str(encoder.returncode)}")
print(f"  done in {cuda_seconds:.1f} s ({frames} frames)")
print(f"  GPU thread time: waiting for decoded frames {wait_input / cuda_seconds * 100:.0f}%, "
      f"waiting for a free output buffer {wait_output / cuda_seconds * 100:.0f}%, "
      f"blocked handing frames to the encoder {wait_encoder / cuda_seconds * 100:.0f}%")

# --- Results ---
label = "with encoding" if with_encoding else "no encoding"
table = pd.DataFrame([
    (f"Video2X ncnn Vulkan ({label})", ncnn_seconds, expected_frames / ncnn_seconds),
    (f"PyTorch/CUDA fp16 pipeline ({label})", cuda_seconds, frames / cuda_seconds),
], columns=["backend", "seconds", "fps"])
table["speedup_vs_ncnn"] = (table["fps"] / table["fps"].iloc[0]).round(2)
table["seconds"] = table["seconds"].round(1)
table["fps"] = table["fps"].round(2)
display(table)

speedup = table["speedup_vs_ncnn"].iloc[1]
print()
if frames != expected_frames:
    print(f"Warning: the CUDA pipeline wrote {frames} frames, but the clip has {expected_frames}.")
if with_encoding:
    print(f"Speedup with encoding: {speedup:.2f}x at this output size.")
    if wait_encoder / cuda_seconds > 0.5:
        print("The GPU thread spent most of its time blocked on the encoder, so encoding and the CPU "
              "colour conversion limit both runs. Re-run with mode = no encoding to compare the GPU paths.")
else:
    if speedup >= 2:
        print(f"GPU-path speedup: {speedup:.1f}x. Worth considering, but only helps if the encoder can keep up.")
    elif speedup <= 1.3:
        print(f"GPU-path speedup: {speedup:.1f}x. Not worth a new backend.")
    else:
        print(f"GPU-path speedup: {speedup:.1f}x. Marginal.")

if with_encoding and compare_quality:
    result = subprocess.run(
        ["ffmpeg", "-i", str(ncnn_out), "-i", str(cuda_out), "-lavfi", "psnr", "-f", "null", "-"],
        capture_output=True, text=True,
    )
    match = re.search(r"average:([0-9.]+|inf)", result.stderr)
    if match:
        print(f"PSNR between the two outputs: {match.group(1)} dB (higher means more similar; "
              "below about 35 dB suggests a colour, ordering or weights mismatch).")
    else:
        print("Could not compute PSNR:", result.stderr[-500:])